# IWISUM - laboratorium 6 - Jakub Worek
---

## Implementacja rozwiązywania "Piętnastki" przy pomocy algorytmu A* wykorzystującym jedną z trzech heurystyk

Dostępne heurystyki:
- hamming
- manhattan
- linear conflict

In [17]:
import heapq
import time

class Node:
    def __init__(self, state, parent=None, action=None, g=0, h=0):
        self.state = state      # Stan planszy (1D krotka)
        self.parent = parent    # Wskaźnik do węzła nadrzędnego
        self.action = action    # Ruch, który doprowadził do tego stanu
        self.g = g              # Koszt g (odległość od startu)
        self.h = h              # Koszt h (heurystyka)
        self.f = g + h          # Funkcja f

    def __lt__(self, other):
        # Definicja porównywania dla kolejki priorytetowej (Min-Heap)
        return self.f < other.f

def get_neighbors(state, n):
    """Algorytm 3: Generowanie sąsiadów"""
    neighbors =[]
    blank_idx = state.index(0)
    r, c = blank_idx // n, blank_idx % n

    # Kierunki: góra, dół, lewo, prawo
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in directions:
        new_r, new_c = r + dr, c + dc
        if 0 <= new_r < n and 0 <= new_c < n:
            new_idx = new_r * n + new_c
            # Zamiana pustego pola z kafelkiem
            new_state = list(state)
            new_state[blank_idx], new_state[new_idx] = new_state[new_idx], new_state[blank_idx]
            neighbors.append(tuple(new_state))

    return neighbors

def is_solvable(state, n):
    """Sprawdza czy stan początkowy jest rozwiązywalny za pomocą liczby inwersji."""
    inversions = 0
    state_no_zero =[x for x in state if x != 0]
    for i in range(len(state_no_zero)):
        for j in range(i + 1, len(state_no_zero)):
            if state_no_zero[i] > state_no_zero[j]:
                inversions += 1

    if n % 2 != 0:
        # Dla N nieparzystego (np. 3x3) plansza jest rozwiązywalna, gdy liczba inwersji jest parzysta
        return inversions % 2 == 0
    else:
        # Dla N parzystego (np. 4x4) sprawdzenie wymaga wiedzy o rzędzie pustego pola
        blank_row_from_bottom = n - (state.index(0) // n)
        if blank_row_from_bottom % 2 == 0:
            return inversions % 2 != 0
        else:
            return inversions % 2 == 0

In [18]:
def h_hamming(state, goal):
    """Heurystyka 1: Liczba błędnie położonych kafelków (Hamming Distance)"""
    return sum(1 for i in range(len(state)) if state[i] != 0 and state[i] != goal[i])

def get_goal_positions(goal, n):
    # Prekalkulacja pozycji docelowych dla szybszego liczenia Manhattanu
    return {val: (i // n, i % n) for i, val in enumerate(goal)}

def h_manhattan(state, goal_pos_map, n):
    """Heurystyka 2: Odległość Manhattan"""
    dist = 0
    for i, val in enumerate(state):
        if val != 0:
            r, c = i // n, i % n
            gr, gc = goal_pos_map[val]
            dist += abs(r - gr) + abs(c - gc)
    return dist

def h_linear_conflict(state, goal_pos_map, n):
    """Heurystyka 3: Algorytm 2 - Manhattan + Linear Conflict"""
    manhattan = h_manhattan(state, goal_pos_map, n)
    linear_conflicts = 0

    # Konflikty w wierszach
    for r in range(n):
        for c1 in range(n):
            val1 = state[r * n + c1]
            if val1 == 0: continue
            gr1, gc1 = goal_pos_map[val1]

            if gr1 == r: # Jeśli kafel należy do tego wiersza
                for c2 in range(c1 + 1, n):
                    val2 = state[r * n + c2]
                    if val2 == 0: continue
                    gr2, gc2 = goal_pos_map[val2]

                    # Obydwa należą do tego wiersza, ale val1 ma być dalej po prawej niż val2
                    if gr2 == r and gc1 > gc2:
                        linear_conflicts += 1

    # Konflikty w kolumnach
    for c in range(n):
        for r1 in range(n):
            val1 = state[r1 * n + c]
            if val1 == 0: continue
            gr1, gc1 = goal_pos_map[val1]

            if gc1 == c: # Jeśli kafel należy do tej kolumny
                for r2 in range(r1 + 1, n):
                    val2 = state[r2 * n + c]
                    if val2 == 0: continue
                    gr2, gc2 = goal_pos_map[val2]

                    # Obydwa należą do tej kolumny, ale val1 ma być niżej niż val2
                    if gc2 == c and gr1 > gr2:
                        linear_conflicts += 1

    return manhattan + 2 * linear_conflicts

In [19]:
def astar(start, goal, heuristic_type, n):
    """Algorytm 1: A*"""
    goal_pos_map = get_goal_positions(goal, n)

    def get_h(state):
        if heuristic_type == "hamming":
            return h_hamming(state, goal)
        elif heuristic_type == "manhattan":
            return h_manhattan(state, goal_pos_map, n)
        elif heuristic_type == "linear_conflict":
            return h_linear_conflict(state, goal_pos_map, n)
        return 0

    start_node = Node(state=start, g=0, h=get_h(start))
    open_queue =[]
    heapq.heappush(open_queue, start_node)

    # Słownik śledzący najlepsze znalezione g dla danego stanu
    best_g = {start: 0}
    nodes_expanded = 0

    while open_queue:
        current_node = heapq.heappop(open_queue)

        # Odrzuć zdezaktualizowane węzły z kolejki
        if current_node.g > best_g.get(current_node.state, float('inf')):
            continue

        nodes_expanded += 1

        # Sprawdzenie warunku końcowego
        if current_node.state == goal:
            # Odtworzenie ścieżki
            path =[]
            curr = current_node
            while curr:
                path.append(curr.state)
                curr = curr.parent
            return path[::-1], nodes_expanded

        # Algorytm 3: Generowanie i przetwarzanie sąsiadów
        for neighbor_state in get_neighbors(current_node.state, n):
            tentative_g = current_node.g + 1

            if tentative_g < best_g.get(neighbor_state, float('inf')):
                best_g[neighbor_state] = tentative_g
                h = get_h(neighbor_state)
                new_node = Node(state=neighbor_state, parent=current_node, g=tentative_g, h=h)
                heapq.heappush(open_queue, new_node)

    return None, nodes_expanded

In [20]:
def print_board(state, n):
    for i in range(n):
        print(" ".join(f"{str(x):>2}" if x != 0 else "  " for x in state[i*n : (i+1)*n]))
    print()

def compare_heuristics(start_board, goal_board, n=3):
    if not is_solvable(start_board, n):
        print("BŁĄD: Ta konfiguracja początkowa jest nierozwiązywalna!")
        return

    heuristics = ["hamming", "manhattan", "linear_conflict"]

    print("STAN POCZĄTKOWY:")
    print_board(start_board, n)
    print("STAN DOCELOWY:")
    print_board(goal_board, n)

    print("-" * 65)
    print(f"{'Heurystyka':<18} | {'Długość ścieżki':<15} | {'Odwiedzone węzły':<16} | {'Czas (sek)':<10}")
    print("-" * 65)

    for h in heuristics:
        start_time = time.time()
        path, expanded = astar(start_board, goal_board, h, n)
        end_time = time.time()

        elapsed = end_time - start_time
        path_len = len(path) - 1 if path else "Brak"

        print(f"{h:<18} | {path_len:<15} | {expanded:<16} | {elapsed:.4f}")

## Proste testy

In [21]:
n = 3
goal_state = (1, 2, 3, 4, 5, 6, 7, 8, 0)
start_state = (7, 2, 4, 5, 0, 6, 8, 3, 1)

compare_heuristics(start_state, goal_state, n)

STAN POCZĄTKOWY:
 7  2  4
 5     6
 8  3  1

STAN DOCELOWY:
 1  2  3
 4  5  6
 7  8   

-----------------------------------------------------------------
Heurystyka         | Długość ścieżki | Odwiedzone węzły | Czas (sek)
-----------------------------------------------------------------
hamming            | 20              | 3279             | 0.0341
manhattan          | 20              | 194              | 0.0015
linear_conflict    | 20              | 154              | 0.0022


In [22]:
n_15 = 4
goal_state_15 = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0)
start_state_15 = (1, 2, 3, 4, 5, 6, 0, 8, 9, 10, 7, 11, 13, 14, 15, 12)

compare_heuristics(start_state_15, goal_state_15, n_15)

STAN POCZĄTKOWY:
 1  2  3  4
 5  6     8
 9 10  7 11
13 14 15 12

STAN DOCELOWY:
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15   

-----------------------------------------------------------------
Heurystyka         | Długość ścieżki | Odwiedzone węzły | Czas (sek)
-----------------------------------------------------------------
hamming            | 3               | 4                | 0.0001
manhattan          | 3               | 4                | 0.0001
linear_conflict    | 3               | 4                | 0.0002


## Generowanie puzli i bardziej złożone testy

In [23]:
import random

def generate_solvable_puzzle(n, moves=40):
    """
    Generuje rozwiązywalny stan N-Puzzle poprzez wykonanie 'moves'
    losowych, legalnych ruchów wstecz od stanu docelowego.

    :param n: Rozmiar planszy (3 dla 8-puzzle, 4 dla 15-puzzle)
    :param moves: Liczba losowych przesunięć. Im więcej, tym trudniejsza plansza.
    """
    # Zbuduj stan docelowy: (1, 2, ..., N*N-1, 0)
    state = list(range(1, n*n)) + [0]

    last_blank_idx = -1  # Pamiętamy poprzednią pozycję zera, żeby unikać cofania ruchu

    for _ in range(moves):
        blank_idx = state.index(0)
        r, c = blank_idx // n, blank_idx % n

        valid_swaps = []
        directions =[(-1, 0), (1, 0), (0, -1), (0, 1)]

        # Szukamy legalnych sąsiadów dla pustego pola
        for dr, dc in directions:
            new_r, new_c = r + dr, c + dc
            if 0 <= new_r < n and 0 <= new_c < n:
                new_idx = new_r * n + new_c
                # Nie chcemy od razu wracać na pole, z którego przyszliśmy w poprzednim kroku
                if new_idx != last_blank_idx:
                    valid_swaps.append(new_idx)

        # W skrajnym przypadku (brak innych opcji) zezwól na jakikolwiek ruch
        if not valid_swaps:
            last_blank_idx = -1
            continue

        # Wybierz losowy legalny ruch
        chosen_idx = random.choice(valid_swaps)

        # Zamień puste pole z kafelkiem
        state[blank_idx], state[chosen_idx] = state[chosen_idx], state[blank_idx]
        last_blank_idx = blank_idx  # Zapisz skąd przyszliśmy

    return tuple(state)

def get_goal_state(n):
    """Pomocnicza funkcja generująca stan docelowy dla danego N."""
    return tuple(list(range(1, n*n)) + [0])

In [24]:
n_3 = 3
goal_3 = get_goal_state(n_3)
start_3 = generate_solvable_puzzle(n_3, moves=100)

print("--- TEST DLA 3x3 ---")
compare_heuristics(start_3, goal_3, n_3)

--- TEST DLA 3x3 ---
STAN POCZĄTKOWY:
 8  3  1
 6  2  7
 4  5   

STAN DOCELOWY:
 1  2  3
 4  5  6
 7  8   

-----------------------------------------------------------------
Heurystyka         | Długość ścieżki | Odwiedzone węzły | Czas (sek)
-----------------------------------------------------------------
hamming            | 22              | 6333             | 0.0507
manhattan          | 22              | 484              | 0.0039
linear_conflict    | 22              | 247              | 0.0038


In [25]:
n_4 = 4
goal_4 = get_goal_state(n_4)
start_4 = generate_solvable_puzzle(n_4, moves=30)

print("\n--- TEST DLA 4x4 ---")
compare_heuristics(start_4, goal_4, n_4)


--- TEST DLA 4x4 ---
STAN POCZĄTKOWY:
 9  5     8
 1  4  3  7
10  6  2 11
13 14 15 12

STAN DOCELOWY:
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15   

-----------------------------------------------------------------
Heurystyka         | Długość ścieżki | Odwiedzone węzły | Czas (sek)
-----------------------------------------------------------------
hamming            | 26              | 101235           | 2.4085
manhattan          | 26              | 1974             | 0.0325
linear_conflict    | 26              | 1333             | 0.0404
